# Phase 6, track M (Mirul): M3, extra GONG photos + pseudo-labels (T4, no training)

**What this notebook does, in plain words:** it downloads ~3,000 extra H-alpha photos of the Sun from the public NSO
GONG archive (the same telescopes and the same kind of JPEG as the competition photos), and lets Kimee's best single
model (`x101_2048_m28`) draw filaments on them. Only masks the model is very sure of (score ≥ 0.85) are kept: on val,
84% of such masks are real filaments. These "pseudo-labels" are extra training material for Syaz's S5.

**Rules kept (HANDOVER §3, plan M3):** no photo from any day of a val 27-day block, none from a test day ±1 day
(the host only asks ±1 hour), one photo per day, train years 2011–2022 only, stations B C L M T U balanced.
No external labels are used — every mask comes from our own model.

| Step | What | Time (estimate) |
|---|---|---|
| A | list the archive, pick 3,000 frames → `manifest.csv` | ~30–45 min (network) |
| B | download + quality check (drops cloudy / cut / blurred frames) | ~15–25 min |
| C | probe: 3 images, shows memory + speed | ~3 min |
| D | archive copies of the 141 val images → val PQ (do the lower-quality JPEGs hurt?) | ~10 min |
| E | predict all photos in parts of 500 (a disconnect only loses one part) | ~1.5–2.5 h |
| F | pseudo-labels `pseudo_labels_v1.json` + save | ~10 min |

**Where things go:** everything is saved in **your own** Drive folder `MyDrive/m3_mirul/` (writes into the team
shortcut failed in M2, HANDOVER §7). At the end the notebook tries to copy the results into the team folder
`solar_filament/external/` and checks that they really arrived. Kimee's checkpoint is only **read**.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Keep the tab open.
**If Colab disconnects:** run all again; finished steps and parts are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive
Needs the shortcut to Kimee's `solar_filament` folder in your My Drive (`HANDOVER.md` §0.6), for the checkpoint.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
MODEL = "x101_2048_m28"
FLAGS = "--backbone x101 --resolution 2048 --mask-res 28"     # must match how it was trained
CKPT = f"{DRIVE_DIR}/output/detectron2/{MODEL}/model_final.pth"
assert os.path.exists(CKPT), f"Checkpoint not found: {CKPT} (add the team Drive shortcut, HANDOVER.md section 0.6)"
SAVE = "/content/drive/MyDrive/m3_mirul"       # your own Drive: the place this notebook saves to
TEAM_EXT = f"{DRIVE_DIR}/external"             # copied there at the end (checked)
os.makedirs(SAVE, exist_ok=True)
EXT = "/content/ext"                           # local scratch (fast)
OUT_DIR = "/content/out"
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Reading", CKPT, "| saving to", SAVE)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5–8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~30 s) and helpers

In [ ]:
import glob, shutil, subprocess, time

def sh(cmd):
    """Run one command (output shown live); stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

def save(path):
    """Copy one file to SAVE and check it really arrived (same size)."""
    dst = os.path.join(SAVE, os.path.basename(path))
    shutil.copy(path, dst)
    assert os.path.getsize(dst) == os.path.getsize(path), f"Drive copy failed: {dst}"
    print("saved", dst)

sh("python -m pytest -q -p no:warnings tests/test_external.py tests/test_pipeline_flags.py")

## Step A: pick 3,000 frames (`manifest.csv`, ~30–45 min)
Reads one directory page per day from the archive (4 at a time). Resumes if interrupted.

In [ ]:
MANIFEST = f"{SAVE}/manifest.csv"      # written straight to your Drive, so a disconnect loses nothing
sh(f"python src/external/download_gong.py manifest --n 3000 --out '{MANIFEST}'")
os.makedirs("external", exist_ok=True)
shutil.copy(MANIFEST, "external/manifest.csv")
sh("python -m pytest -q -p no:warnings tests/test_external.py -k manifest")   # no excluded day in it

## Step B: download + quality check (~15–25 min)
Keeps frames whose disk size, brightness and sharpness lie inside the train photos' range (clouds, cut-off disks and
blur fall outside). Kept → `images/`, others → `rejected/`, numbers in `qc.csv`. The kept photos are zipped to your
Drive once, so a restart unzips them instead of downloading again.

In [ ]:
ZIP = f"{SAVE}/external_images_v1.zip"
if os.path.exists(ZIP) and not os.path.exists(f"{EXT}/images"):
    sh(f"mkdir -p {EXT} && unzip -q '{ZIP}' -d {EXT} && cp '{SAVE}/qc.csv' {EXT}/")
if not os.path.exists(ZIP):
    sh(f"python src/external/download_gong.py download --manifest external/manifest.csv --out-dir {EXT}")
    sh(f"cd {EXT} && zip -q -r -0 external_images_v1.zip images")
    save(f"{EXT}/external_images_v1.zip")
    save(f"{EXT}/qc.csv")
print(len(os.listdir(f"{EXT}/images")), "images kept,", len(os.listdir(f"{EXT}/rejected")) if os.path.exists(f"{EXT}/rejected") else "?", "rejected")

## Step C: probe (3 images)
Look at `Peak GPU memory` (must be below ~14 GB) and `s/img`. Step E ≈ s/img × number of kept images.
If this fails with "CUDA out of memory", stop and tell Claude.

In [ ]:
sh(f"python run_pipeline.py --step dump --name probe_{MODEL} {FLAGS} --weights '{CKPT}' --image-dir {EXT}/images --dry-run")

## Step D: archive copies of the val images → val PQ (~10 min)
Same 141 val frames, but downloaded from the archive (JPEG quality ~75 instead of ~90). Compare with the same model
on the competition copies: **official PQ 0.4355 at thr 0.6**. Within ~0.01 → the extra photos are fine to use.

In [ ]:
VALC = f"{EXT}/valcopies"
raw_valc = f"raw_preds_{MODEL}_valcopies.json.gz"
if not os.path.exists(f"{SAVE}/{raw_valc}"):
    sh(f"python src/external/download_gong.py valcopies --out-dir {VALC}")
    sh(f"python run_pipeline.py --step dump --name {MODEL} {FLAGS} --weights '{CKPT}' --image-dir {VALC}")
    save(f"{OUT_DIR}/{raw_valc}")
sh(f"python src/external/make_pseudo_labels.py score-val --raw '{SAVE}/{raw_valc}' --thr 0.6")

## Step E: predict all kept photos, in parts of 500 (~1.5–2.5 h)
Each finished part is saved to your Drive at once; after a disconnect, finished parts are skipped.

In [ ]:
imgs = sorted(os.listdir(f"{EXT}/images"))
PART = 500
parts = []
for k in range(0, len(imgs), PART):
    name = f"part{k // PART:02d}"
    parts.append(name)
    raw = f"raw_preds_{MODEL}_{name}.json.gz"
    if os.path.exists(f"{SAVE}/{raw}"):
        print(name, "already done, skipped.")
        continue
    d = f"{EXT}/parts/{name}"
    os.makedirs(d, exist_ok=True)
    for f in imgs[k:k + PART]:
        if not os.path.exists(f"{d}/{f}"):
            os.symlink(f"{EXT}/images/{f}", f"{d}/{f}")
    sh(f"python run_pipeline.py --step dump --name {MODEL} {FLAGS} --weights '{CKPT}' --image-dir {d}")
    save(f"{OUT_DIR}/{raw}")
print("All parts done:", parts)

## Step F: pseudo-labels (score ≥ 0.85, usual post-processing) and save

In [ ]:
raws = " ".join(f"'{SAVE}/raw_preds_{MODEL}_{p}.json.gz'" for p in parts)
sh(f"python src/external/make_pseudo_labels.py labels --raw {raws} --out {OUT_DIR}/pseudo_labels_v1.json --thr 0.85")
save(f"{OUT_DIR}/pseudo_labels_v1.json")

# Copy to the team folder and check (writes through the shortcut failed in M2)
try:
    os.makedirs(TEAM_EXT, exist_ok=True)
    for f in ["manifest.csv", "qc.csv", "external_images_v1.zip", "pseudo_labels_v1.json"] + \
             [f"raw_preds_{MODEL}_{p}.json.gz" for p in parts]:
        shutil.copy(f"{SAVE}/{f}", TEAM_EXT)
    time.sleep(30)
    missing = [f for f in ["pseudo_labels_v1.json", "external_images_v1.zip"] if not os.path.exists(f"{TEAM_EXT}/{f}")]
    print("Team folder:", "OK" if not missing else f"MISSING {missing}")
except Exception as e:
    print("Copy to the team folder failed:", e)
print("If the team folder is not OK: on drive.google.com share MyDrive/m3_mirul with Syaz and Kimee instead.")

## 5. Send results back
Download these small files into the repo folder **`Result Phase 6/Mirul/`** on your laptop and tell Claude
(also copy the `Peak GPU memory`, `s/img`, `[Val copies]` and `[Pseudo-labels]` lines):
`manifest.csv` (Claude moves it to `external/` in git), `qc.csv`, `raw_preds_x101_2048_m28_valcopies.json.gz`.
The images zip, raw parts and `pseudo_labels_v1.json` stay on Drive (too big for git).
Then **disconnect the runtime** (Runtime → Disconnect and delete runtime).

In [ ]:
from google.colab import files
for f in ["manifest.csv", "qc.csv", f"raw_preds_{MODEL}_valcopies.json.gz"]:
    if os.path.exists(f"{SAVE}/{f}"):
        files.download(f"{SAVE}/{f}")
    else:
        print("not found (not finished?):", f)